### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="sepsis_prediction",
    dataset_year="2019",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/salikhussaini49/prediction-of-sepsis", # alt https://physionet.org/content/challenge-2019/1.0.0/
    download_description="""
We download the data from Kaggle as the link for the data from the original PhysioNet challenge seems to be down.

kaggle datasets download -d salikhussaini49/prediction-of-sepsis && unzip prediction-of-sepsis.zip all_files && rm prediction-of-sepsis.zip
mkdir -p local-data-warehouse/sepsis_prediction && mv all_files local-data-warehouse/sepsis_prediction/
""",
    # References
    academic_reference_bibtex="""@article{reyna2020early,
  title={Early prediction of sepsis from clinical data: the PhysioNet/Computing in Cardiology Challenge 2019},
  author={Reyna, Matthew A and Josef, Christopher S and Jeter, Russell and Shashikumar, Supreeth P and Westover, M Brandon and Nemati, Shamim and Clifford, Gari D and Sharma, Ashish},
  journal={Critical care medicine},
  volume={48},
  number={2},
  pages={210--217},
  year={2020},
  publisher={LWW}
}
""",
    academic_reference_bibtex_key="reyna2020early",
    license="ODC Open Database License", # CC BY-NC-SA 4.0 on Kaggle...
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We start with all files from Kaggle.

The original data has one file per user that was already preprocessed to one .csv file by the competition creators. Here we start with the preprocessed single .csv file. The data is non-IID in nature based on the groups of patients from different hospitals. The data that is grouped per patient ("Patient_ID"). These groups are also temporal in nature, but this temporal dependency is irrelevant as teh task is to predicts for one full patient (i.e., no refitting given patient information).
In the original competition, one had to predict for unseen patients from the existing hosptials and also for a new hidden hospital. In the public data, we only have two hospitals, (A) and (B). Given the limited data, we decide not to simulate a domain shift as we could not "train" for domain shift. Thus, we simulate only a normal grouped non-IID scenario. That is, we use all patients from both hospitals for training and testing, but ensure that the splits are grouped by patient ID. Thus, we simulate what would happen if someone trains a model on data from two hospitals and uses this to predict for other patients from these hospitals. This decision is also amplifed by the large gap in performance for the unseen hospital in the competition (Report, Table 3), clearly pointing to a domain shift that is out-of-scope for this task. Note, we do not have temporal information of the order of patients, thus, we cannot simulate to only predict for patients "from the future". We believe this does not introduce any data leakage for this dataset.

- Patients with an ID larger than 100_000 are from hospital (B), while patients with an ID smaller than 100_000 are from hospital (A). We add this indicator into our data and then reset the Patient_IDs to be continuous increasing integers.
- The data consists of a lot of missing values due to missing measurements.
- Note, hour is a reset time index per patient. ICULOS is similar, but with an offset. We keep both as this can show that we have truncated data for a patient.
- We reverse the ordinal encoding of Gender.
- We mark features as categorical where appropriate.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="SepsisLabel",
    problem_type="binary_classification",
    objective_metric_name="PhysioNet2019UtilityFunction", # https://github.com/physionetchallenges/evaluation-2019/blob/master/evaluate_sepsis_score.py
    stratify_on="SepsisLabel",
    group_on=["Patient_ID", "Hospital"],
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "all_files" / "Dataset.csv")
print("Loaded data shape:", df.shape)

# Add Hospital Indicator
df["Hospital"] = "Hospital_A"
df.loc[df["Patient_ID"] > 100_000, "Hospital"] = "Hospital_B"
# Reset Patient IDs to be continuous (after remapping, IDs higher than 20k are from Hospital B)
codes, _ = pd.factorize(df["Patient_ID"])
df["Patient_ID"] = codes

df["Gender"] = df["Gender"].replace({0: "Female", 1: "Male"})

as_cat_type = ["Hospital", "SepsisLabel", "Patient_ID", "Gender", "Unit1", "Unit2"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.drop(columns=["Unnamed: 0"])

# Remove order of patients so that method figure this our themselves (remove order by "Patient_ID", "Hour")
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

Loaded data shape: (1552210, 44)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
    duplicate_column_check=False, # We know the data has unique columns
)


#### Dataset Overview
Rows: 1,552,210
Columns: 44

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,FiO2,pH,PaCO2,SaO2,AST,BUN,Alkalinephos,Calcium,Chloride,Creatinine,Bilirubin_direct,Glucose,Lactate,Magnesium,Phosphate,Potassium,Bilirubin_total,TroponinI,Hct,Hgb,PTT,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID,Hospital
0,14,68.0,98.0,36.80,124.0,88.00,66.0,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.00,Male,NaN,NaN,-104.13,15,0,37206,Hospital_B
1,48,98.0,100.0,NaN,151.0,111.00,86.0,16.0,29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57.00,Female,NaN,NaN,-2.80,49,0,24182,Hospital_B
2,4,75.0,100.0,35.39,88.0,56.00,NaN,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,82.33,Male,NaN,NaN,-2.66,6,0,4273,Hospital_A
3,7,95.0,97.0,NaN,118.0,80.67,NaN,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.65,Female,1.0,0.0,-0.02,8,0,17236,Hospital_A
4,11,46.0,99.0,NaN,169.0,111.00,77.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,69.00,Male,1.0,0.0,-5.06,12,0,24773,Hospital_B


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Unit1,category,611960,39.43,2,"0.0, 1.0"
1,Unit2,category,611960,39.43,2,"1.0, 0.0"
2,Gender,category,0,0.00,2,"Male, Female"
3,SepsisLabel,category,0,0.00,2,"0, 1"
4,Patient_ID,category,0,0.00,40336,"8167, 10102, 26149, 37625, 2194, 34279, 19786, 22913, 36144, 12397"
5,Hospital,category,0,0.00,2,"Hospital_A, Hospital_B"
6,Bilirubin_direct,float64,1549220,99.81,280,"0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 1.0, 0.8, 1.1"
7,Fibrinogen,float64,1541968,99.34,823,"217.0, 180.0, 202.0, 185.0, 219.0, 200.0, 151.0, 214.0, 248.0, 208.0"
8,TroponinI,float64,1537429,99.05,2423,"0.01, 0.03, 0.02, 0.04, 0.05, 0.06, 0.07, 40.0, 0.08, 0.1"
9,Bilirubin_total,float64,1529069,98.51,407,"0.5, 0.6, 0.4, 0.7, 0.8, 0.3, 0.9, 1.0, 1.1, 0.2"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Hour,1552210.0,25.492745,28.882557,0.00,335.00
HR,1398811.0,84.581443,17.325242,20.00,280.00
O2Sat,1349474.0,97.193955,2.936924,20.00,100.00
Temp,525226.0,36.977228,0.770014,20.90,50.00
SBP,1325945.0,123.750465,23.231556,20.00,300.00
MAP,1358940.0,82.400100,16.341750,20.00,300.00
DBP,1065656.0,63.830556,13.956010,20.00,300.00
Resp,1313875.0,18.726498,5.098194,1.00,100.00
EtCO2,57636.0,32.957657,7.951662,10.00,100.00
BaseExcess,84145.0,-0.689919,4.294297,-32.00,100.00


In [7]:
# Categorical Feature Statistics
cat_stats

value    count    pct
column      rank                            
Gender      1           Male   868103  55.93
            2         Female   684107  44.07
Hospital    1     Hospital_A   790215  50.91
            2     Hospital_B   761995  49.09
Patient_ID  1           8167      336   0.02
            2          10102      336   0.02
            3          26149      336   0.02
            4          37625      336   0.02
            5           2194      336   0.02
SepsisLabel 1              0  1524294  98.20
            2              1    27916   1.80
Unit1       1           <NA>   611960  39.43
            2            0.0   473349  30.50
            3            1.0   466901  30.08
Unit2       1           <NA>   611960  39.43
            2            1.0   473349  30.50
            3            0.0   466901  30.08

In [8]:
# Target Distribution
target_df

,count,pct
SepsisLabel,,
0,1524294,98.2
1,27916,1.8


## Task Curation

In [9]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from sklearn.model_selection import StratifiedGroupKFold

splits = {0: {}}

# We create one stratified grouped split based on Patient_IDs and Hospitals
# to get a good representation of both hospitals in train and test.
df["group_col"] = df[task_mold.group_on[0]].astype(str) + "_" + df[task_mold.group_on[1]].astype(str)

sklearn_splits = StratifiedGroupKFold(n_splits=6, random_state=42, shuffle=True).split(
    X=df,
    y=df[task_mold.target_column_name],
    groups=df["group_col"],
)
for fold_idx, (train_index, test_index) in enumerate(sklearn_splits):
    # Print len, target col count, and group counts
    train_data = df.iloc[train_index]
    test_data = df.iloc[test_index]
    train_data_h_a = train_data[train_data[task_mold.group_on[1]] == "Hospital_A"]
    train_data_h_b = train_data[train_data[task_mold.group_on[1]] == "Hospital_B"]
    test_data_h_a = test_data[test_data[task_mold.group_on[1]] == "Hospital_A"]
    test_data_h_b = test_data[test_data[task_mold.group_on[1]] == "Hospital_B"]

    print(f"""Train N: {len(train_index)}, Test N: {len(test_index)}
    Target Distribution:
    \tTrain target distribution: {df.iloc[train_index][task_mold.target_column_name].value_counts(normalize=True).to_dict()}
    \tTest target distribution: {df.iloc[test_index][task_mold.target_column_name].value_counts(normalize=True).to_dict()}
    Group Distribution {task_mold.group_on[0]}:
    \tTrain: {len(train_data_h_a[task_mold.group_on[0]].unique())} (A) vs {len(train_data_h_b[task_mold.group_on[0]].unique())} (B)
    \tTest: {len(test_data_h_a[task_mold.group_on[0]].unique())} (A) vs {len(test_data_h_b[task_mold.group_on[0]].unique())} (B)
    Group Distribution {task_mold.group_on[1]}:
    \tTrain: {len(train_data_h_a)} (A) vs {len(train_data_h_b)} (B)
    \tTest: {len(test_data_h_a)} (A) vs {len(test_data_h_b)} (B)
    """
    )
    splits[0][fold_idx] = (train_index.tolist(), test_index.tolist())

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create stratified grouped 6-fold split (ca. 250k test instances) based on Patient_IDs and Hospitals. The label, hospital, and patient groups are equally represented in all train and test sets.",
    splits=splits
)
df = df.drop(columns=["group_col"]) # Remove helper column again

Train N: 1294409, Test N: 257801
    Target Distribution:
    	Train target distribution: {0: 0.981662673853473, 1: 0.018337326146527104}
    	Test target distribution: {0: 0.9837859434214763, 1: 0.016214056578523744}
    Group Distribution Patient_ID:
    	Train: 16970 (A) vs 16641 (B)
    	Test: 3366 (A) vs 3359 (B)
    Group Distribution Hospital:
    	Train: 659260 (A) vs 635149 (B)
    	Test: 130955 (A) vs 126846 (B)
    
Train N: 1291661, Test N: 260549
    Target Distribution:
    	Train target distribution: {0: 0.98200921139525, 1: 0.017990788604750008}
    	Test target distribution: {0: 0.9820456037060208, 1: 0.01795439629397925}
    Group Distribution Patient_ID:
    	Train: 16945 (A) vs 16677 (B)
    	Test: 3391 (A) vs 3323 (B)
    Group Distribution Hospital:
    	Train: 657510 (A) vs 634151 (B)
    	Test: 132705 (A) vs 127844 (B)
    
Train N: 1294381, Test N: 257829
    Target Distribution:
    	Train target distribution: {0: 0.9821659928568173, 1: 0.017834007143182725}
 

## Export

In [10]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c14ab-0746-76d1-bc27-4224de90101d
571fcf4229434839024bfe332d63e10b041ec50cbe1d6dcf7ed88d107aa45e96
